# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq1PWtz27aW3/UrsOx0SrYyKzmPptqqs0nstJl1Em+Stnev6+FAIigx5qt8yFYy2b91v99ftuccACT4kp3c3Exr8QEcAAfnjQPQsqwXaZnmCxbAj1047OhnFvMo4uzeCbN/rZJ9xZN7J0fH7Ht2HO/ofSluyirnePkkEokvcncy+a1ImS9YLtZVXqTFYsLYt+xt6qcFi+D/OPUF/v5VCZ8nbM3zDcd3ImHQ1i/nvzGOFzvxntmZyIMwYi8e/+35i1fse4DE2OOzt6+cKeNFyJKUbfmeiQLKcYRQ5oJVCachsD2CKcJNFcIL4VI3zqChTECvWCFYBl3E1hPV9uvHL5i95uutwP5nfBMmABUuz8KkunFYJPYwxpR6gSPg+Xob7mTXM57zCMa1YApoDh1RY2Ulz33oithUUL2QPTmNWJnzFX8Hr0WcpVCd7TiC+t4Xu++LbcxsPyzWKT6CnjlqABJF1Pu0FBvB1mmCTa154gMe/1NPBLOfQrnvqCqUyQEzIqGu665STRNFbCMSQOQ6TJNmNhq85aKoopLmCprfVDgoKogYl2iTXWUwGpiYYou4i/RsNGh7VxWlKBBTmSjDHIvEYRGrgqJgYVKUPIH/ROpOLMuahIigEjsMNJVDX92gAsKDojBB60C/36zrqyhd6WvsSRTWt2Gqr94VaaKv00Jf5UJfFduqDKP6rloByteiqEuW21xwP0w29YMwriu/DzMgXTHR90kVZ3vsbpJNgjyN2fnzM6bePY/5Rkzlz6usmMgCLmuGHYSbyeTX09enbAlddTNebl0/zBMeC1vf81WBv7bnYcOe5zgTmD/v7NXT/4ZadWfds3R9ZTuT35+f/vEGXtgWtJaU1pRZK76+wt9IBHSfh5ttaTmTyeQr9gaoNw+BxBbQLY6UUrBNCMwDU8lSFvCowMGlLAO+Bjooq3gFTCAiIJZ8F/ppPqHxucDM3vMXj3859c6f/+30DLvw8L43m83w/wm+ffP8BMd5PLv/CJr2RcAKHgiPmrXDeOMsiJaAMH7nEXQJOD1Jc5BV4XtOxCg7iDOGb+2SxzxJp8iN8DfAomXquEhYCCcMoDrQRSHJbk1tqNlw6a9qEP/lPATih3YrcZrnaW5bwMhKEODoARlGD3bUP8AgVr2esi0MC4C7Rfhe6Lav2U/s3jFDlqGrQ22d1ZAFCqYYhCDwI0g0EKbQqh2HSRinCC67cVSz1MS3APxnNjwBn9HgJgdpI5oGYn5j4/AcaETPYAMWRqzGvU6zve2YL9xyC4SS8DCybV1zWsNw9DScPX759O+v3tQNYk0UrnLmEiDi1788QZKFn8eWM9x2shN5aasiVCIXIEUSfK0IbR0JnlSZl1ZlVpWFfSVEtnw0A9oJEy/IhfA2q+V81lDgE5CsnMSaIR4BTWwXincoKUNSUTXqFK+UnJFwr4mwzPdNp/0C+lwA6wvf9pFgmY+DRKHm4p+a5d+lYWJL8eC++u2td/L8NeDgW8txEEm6VFiAqLB9p0E8jgSbIAEHgqS48qoCuasNy3GpIOj7b7+9N6trX29BvmAngQiYDerG9guceUQW0jFV+slEmTEhND5o24fOpZk9c1pvVI/yGDS5sP0pCzfA2sITSI7F8m1eiXYFY5jiJixgynxQepYL4tfqtIr/oGwu4nQnzGKtUp+PGnGzFlnJTukHdGjTesZBaUwm6wh+2WkCdoWQL5HkPA+YtvQ8uxBRMEW1HixfpgnogyjdLLM8TEpjIFjIzaCDWA6Rrbrli1KsS4O3qCBAgKLwt/0YULDCF9hM+02x5ZkYf+PFuwMvy4MVR15nHAY4/sa7ChMkF8Um9ct36QqZ5IPF1yXaYNaCAZdaIvHWacTx7mO7ggfWHvDkNVR6BrqqaesrUNtogAHbCj9EDrXxAmySdBdGDplLIP1At4M6W4MNXCoObsMXwNLYH1Iv0P4jF/ujDGS4v/eAHqykdQYPHs7cXh8BiIe0bGj5YRZXXXbRhDEIuCVFzK65VebzUthYHqiC+3aaAd+2W3UMCTFOzDVBt9qQ9h8g4GOX1Li/7+GcXilUeFmaRlBiHbhvyUY5h/vTG/AewBuxQbN412l+JYD7j0GNScq4sNQz67LpMqAj5lcwdXnRQxjJBi+9UhKkYT6RFGBHEiIMLgOp/CZE22Wd+uEmRXsX6KxCC0OZzYYjs+e1EzOFl3EWIS3AmxocL/NwBcMB+6gS6DAQTSVo+YPBjZ4QaA+QAuuQkzVciAJQXusGLTLlbHl+uAZx0bxxC1HCWDhoH9tSnAiqEBnKOVgKefKWgg0PonK1xoohN0KBQ9w4VlXzJVQnGhkrBzilFj6Nv0Zb1TQPMD+F08bgEflj/4wWQTX57kbAW5PSLVDNBeG8oxMv2sUu/xWWmEjBdvQv/JM0XTMKzBKKDe9aoE9QKGUFHlxaLG2YBjCbk/J7I0pg0ez0Hsc7QGKL0c5AjZZ1dKD2qdEDJ0edeMUX73ge1VECcDJb3EH+0rbagGbdBHwtvG210t5TAZ6xR/MDnJM3VcCGQQ1ycTkuPdHyytHyasOwHZcG3jcvYMpzeueFZLGNFKthix2VcpEFkOVHyjb9/Q46XJS5Hbjk45HUJlCBbG5Hz4vLUTmOHqgY0BFgI9jWS5J0WQUWfiQEiaI6HPLrswVMacf6UvbzbNKwBdBK7tvUsXYzCSB71npyHYKmIz2ExdHfXEETqyoIBFg9m+XMwd4GfZxIAxRF+TC+VtCUVD32w/vsp5/AlXQGCyrXbzWOdjXCZLBAgvOB9u/KaTCAxgwGAlz8YxhkNNphjp4/pJGKm3Y/yrTkqBmLKrbFDSi3zM7BJKYZdvqWnh1Y55JjVAhKRdWAWRbsgwSm7NWFOw8+sl+e4GurP7TA+mCMgB2xcuHOgo+FIf8VXgiqoU6lYsnCTBgSYoqe+JRBhVVqEIVk2v09H9xLaSWin6m4thYZJ+HbZ1F6/YKXIBiSzTmAjkLDVMS2AEe3lXexNdQ1JTi5CfhVHTpWvQzSCCTwkvpbFcLDwANIsaLvfADp0IDaU/YVewZG/vb3E0C5L9B6CMI1xTNlKPX/5rMbcg9znoUqlDgqfvTwXPDtQTd4AYLe+bEdrz0ebdKlFa873oti97f7THrwfYCDrYy21GebOwiUvmBhNVrQ6gGnKkuTEBoaECl3hP/JsBXB4igbeiVLWBLqNRj2YBHRLKs7Mn7MBzTh5oO/0CQs9+SvtbXaU2RA00x02VuszewHaD3D8+/qfjsYXAFjE+NTFDuFroEzZ5AWBVHB7mEFjyjKWseS2VNAEUaEZUiGlTxehVCAouvaLgXgNSgVkpYBM+jNTHZH+zmqF08pcOW3lKwOcqagohtph3cuBg7BdincdeVzEFRlXEUuEHx67ZXBvWPgT0TagUpJ8qnFV2BbbMElu+qWbokV6emqnj8BCJs8BcS9phhAbjKzOZ/DFlqrxLI2u6DfXpp4m6wCmw0mDxVKQy236wVo23DKB1tuOe29UbRhNU1TeKblno8DJ04imgWqYuZij6JaSX4ktbjjum5H7gxKlU5oQLovhoYYNhtHVfHYP2urqvtHflge7Y6PZkeEABAA/YDRZ8gXiYLbBNcAibTosktj7dn5rLmRswJOp+TWgXlpBXXuOAFDCO17Z3o48Y5GMk52FPn5jNEBYEV7O56HFFEtSl58BvGp2NOdhg/+yadT4CARxrsvS4UNQg5RYmda7BbJgXQamRinP2ufOWft2TpEm7dP27930oamrE/n/8JsteZpiq5VVaCSLVJQwHNa7RyZPRkUradE3g7OR0vbleJmyIQ+x+o9w5l6XFEU1b4wcEGNmVL0sqNWBOAH3GYHLJjhetZlz4su0B+nFbniAN6CIaKi3jP7wzfU+jfYmfqSIMoOfSMtmm8+DlDaQbt39EUvRD2M0MOeRY/ohkVuy/UY9lVvN+0NKrLk1CkMLQ52iVb9vigGBgfpTA6D15F+6O+wUw9u89WXckfuoswPL1GYplvNn2j79Xm0ie8XApiIQpbgWobvpZOMRiQ8NEPZvQYurDQI0FuxLhe3dFT7cGToe+us8lTVjjOHXHOL3aOye54/buOL2WuQYDkuQe3BT0EHo0n0cKyWOEOMDFnK/UBbRuu4Lfk/7aor84EzGH7LzGkgCVo/Mc3RERfYLVPbApRZzvAqRsugozfXPI+rzB6MxrxQCwQRqEIZirk1rKJre7FA57t2VFUz/WWK3xKODhz3MYcmCcFGB69OJiAtaDEijKQbeiXyBKYcZ+rpbyeP2R4IINw0o6Hga7RJ87CMZfh1XZ28fCkTrQBGlkOvc53MwzFXiWPcFjO1onR9N29RBWtajkHLKDFJhworpacdK+kKkpcZFh7fcRgeUKTdIQbp7N/qevUDvkOJOSc5v26zfQzwZF5CIq5VQsEUfOn58ZTBHweuZ1Om/nOc3tp3DdbFP3YYd4q4IorCrBC2Pf/h4ZTdBzD37sHF8WzmUPwvWtpwPWXzH/DPA/hz/OBBtyGgWmDOZBMhnB+h+vxHBDRHiD80gB7CQ8xwmD8cBIMup8T6L5QdhvFKX+zCtVhaOA+WA/5+AgYmCDcQ/Z1AKwU9ZfUwCUSOeoEkkw3ty+e8KtM1L3ARBcFNmV+CilvKlyi4yvlDZzgwZlNuyjKMp5he5TUtFKXIiiXMRrrGRAIvF2BzVagflvPjRyO2IoJYgz1zVSy1xJK31uVUpcZBp5YbgErZIR710yqRMYpt17htCLXYJ+ttDqLzvbCdyYjhgyEcMJVBaSIr30VUfFJkn2ngMWb6pSyNwzL0tflpxG+HFBRIXbg0k2qCW+3SLgcOclrbbnX9MKAJLENSYDmthoncRfSqO82YL+DRa3oyGdGCaiHPHAxKHp14NV5DNlRXUJQgn3YsW6Oy6t7S6JmtVgdN6uuC+yS3xRzLkqak3xucve44P3cNaI2xRJnDJ1PoUCuoFdcu5Wid1dFQvXlX85cVlFo58KatMPDfDsWtrODuwhyX4bGpNN/bY6JKhiGpjIdxxTXHVCpHL39Q8H5Mk0h3YjbGqJjfWpG+/bCL3aoQfmtR5Xt82llrmcm1ln/+Y2itRa63IPt/2CgIiki+2eU89jarby6xuvVJuRlfYgFYp3maa8CZEgiUISkzqbzVRi6NtOQD5t0dEAxtWxrLd7IvizJ3upSzqdUuLRpiJmgnz6+TJzoZqPsqKzBTLPDAWUnAoi06RbccqCHKtlylDVKW4XKpMgvJokwyl+c531PFC3A3QateumB+AfH8BCp0Zg6sRpKsmpZNC0Pja0K9djdp0cw+OJzWKCcf1D8wFPAv0GK6eicoJR7dshx5WiaNV9zP0X7HnJYYrEbRiGxEwOBI6xL7YspuClksSZP3Ik9tzn5m81nLBcAF0puiM5s3YGzczKdsD7/7OcC4KSQGp3TFb/BqXz/bq2cDGCMk5Glm2whzT3DZd2xOcOHXwNpQBi6xNz5MSrtJY/2Wzd35I8Mu/2vY5AMMFKPmXvEXKAcwRSRF23YBevwahBAIBqwKd1t55/SWqaAqaA1UwPZ8dnwfrDP428uGJUVMCD++L8VW8VcjkossCkuv2ApRajWO12C+5BhokGnfU8z1nuLaylRle7fci5NwF/pyT8U2fYd5GCqwJ/c8gEchcDcArh3hajMuXgVp4qdtVyCQLR8UCEbiWL5Z8TYrjHFuJ3eU2uhlc6tRo0SRSDJkCL1znFE+azEB9qvmAkJ7XW5NKWzc5cneNIELsQEKgZ4AErEHPC8p5URFGaeGmkEXOJwySggRYIqitYlJp1FXDAZsV4sSCXnAEe60KJcsw47/r1AlwR0G1cvgM0fo8gxw6QOBY3PQjtPi/yHIQ9VQTuB4jdp/EFLJM7yYX7bQingsZGyRolz4CAkNygFjFRcznKE/gJFn7uz+ZVceYXES1gsSlUAsbFuBvCwWuA6KWxrCDdgZouilsGBtYiCX+Mu2ppbj9MaGvbNRpITQgz/AFEicKYkYeIAyqX4qs3dCyt4BX03YidN0Frd3EKjUBeoNM1uWTrF0tw9NLfBO2omYWAdBgdCRYtdBAO8BHjUwpR53yAyAXODby9oGRuWPHGAIWyXD6SkKK8C94/Qi6IrBAeIXMUtozavxWKBbPMyVgENPYYq7ETzMAivummxinwjl34lnUE+tq+JA8Mav1nBzR0P9GTqsIlcgGiRgz5CJWq9tx8bnvVKD3ekVHrWt91gOrNnWyxgJqX7lopvyRpQdjRoDU/q+h4Xsdllq20UZCZ5/ITHtSiT3QOC7MNl4yvTxKyBStMA9Xd0+UAVn0wNrwgOHI8SYuCdgzg5UWUegCbxtCsyKCpwu0EE6ns1mB9rx0hxdYUko3jrdkucZdX0K9D1izDmgbWcSM86g3yLDAO3KcjrVK/et/LVjwqO4gSHCK7J0YsKl8UBtNVt2eOkTnf56DofXonTPBgLgg6siB4aqhwsYBitybKYlZIkNOdVuEN54cumm6ND3aIUo0tPdKz+In+PDYe6czBfa9SjDItK5Pe4ZsQbNs58bKbMYmnNDdCjGbeQSaPdlI6O6cpJQ2xhxFCVF6aVEHOZKo8jGCJeOb+F8Cn/KNhV0HkwelWvXc8uGPOy4isoQ9cXVAiwBVBVXygqhltywFDFMo7Q7jNh6o170erPUjQQP8TNvjCOwL+Qy44FwPjy4rJPq1Brj4cWAS4f92TWNdF/UGmWnC5Q9a8SeTZAHGm+1XF52o9mJuCntrFnDkG111yKaeca8QkKXDHDfuuhh0GFBiQxD9cpD1QKq2RETSEAo3MDBQsOE7kHwPHKkzaafsJ+WYOhKnDz4vJWdtNyKvNPhNk47ne+PZXCtR8HtrPe0niKqhtd6qNjAek+dCkmvqnr35kBgV8RZuZdp4XbXNZdsBZ1Us034I466UPttGzr6xBA7zQ1cOJ3c4i8eZleirBVp3wyH2g0KGjWU+rF4rCWfOtOD4fdRmFrieQXGt5c0Dls/dNph+/GODYbzwYMYEs1TNaNfxJZVwdRa2qsIb8uaVQIf7YXPDcMrSG0PLgxqmOw/loCpkpelNqQtjCPrvUED9kV/oaDun8kI5Ly0qf4S00bko6tLqW/kBt6RzeekeEgT4VaaK+fAdo2v2O8y9ygXkjrx2g5yOtvgOzwmgkfAtq7rMLmmCVYK7rpgUcq4D6pfdF2g2vnhFB4iLNacUGi7AK9J40kux1sgns822vShGhiTayVToRFH/QTynsosIjm4Afvt1s63eti4aDJZ7csQd6oS3JGyr8S+bcCkWWn6Zyho1EENKPfnhkCtaYROLGjjDdWoSR39UMnBrIyt3hd55QK5kLx0xgvtQCes9iVatReLBfvxBwOD29YOS7+Ks8LGIU5pH7cHw1cbCAYaUtjfultx44fg7JTYwPz4sjED8ypR6JNRO4bcorcJA8OoS+IZdU2hLXndD5N5B2OA3dD+tG85qEak9FY3ZM+qayBi3Q9QVcv58b37bShaSC9p3x60u7S2FS4yhT5upNvSPrPl3H0EL1MQTWI5g4LtyYei6dJ6ysGG4bk1lQ9QweimOW7GrLsESC5xjWdgTEqUqdz+XESh2GGLj0xTmryyTY5uWcTjlc8ZCukFU6hIPAwdhTwaQDr1THlzVBT4eaPx5oPojVS3ulu99QbV4aQayksZlCStdJ/zWhRMuuZfrbgxw+L8wqLr1q5ammFVUt3IovLGLEsEoIrKa1mSrs2CcpESi+GVLNRJgarnUhVs7mXx+t6spCZcj0vdqZHJu1Y3Zp3EkMa0VxNtA9EdgzL6nyosKS+RotuW2U0QPnLjN+m3hREtU6xa85Kjj1gxCxHbmWX6ZopUiWYlYtg2YMm5ZiF6Ypb6OBiRH5aOpiMojXFzNUFF1BtJ4mg/cdBQNyX0sJ3SQicaBiDWJ10ItQ0xYuv0Ti95Rgdt6BOIfKHUpTF9KKLxMIpwrdhg2XKsl9q/lqKN/krZBv8r0Vab48MGZi3nGucchR38Xws5+VPLOfkzHcEPijz6a4q7+qqReep3GEwjCldpGtn1rTOtJaF8IW+cRiRKC1vdOaPQQWJqyKvUmWr5R8/ktdPIPXqqbgy1eKWlFtoOptVgEsZVTaBypyp5gK1zT5qXQFcXFjwCETBmfb3Wh7jg+hZVQtvqanDXawtwIzq6R8g4ncD8we3mJIyKMg9IIFlf/+/X8de+9/WvX7/4+o1nYcr11fCBB2S6dU456ByqQofvtFrHA6Vwm35RVn6YutsyjiyH0puMyqiosNtKZ6E0tZX6k4sK/qIT4MCN6N2Bnsv96R6hfzo2CiwyOgxaxhiKTS0+0erbuQXfiTYqZMuBRZa29yH56GbJBld26tqnmhyFcfTHV+w1Rg9JukRsPpuxr+XePPQp8FwHPPoNM2nQWOCF0Cfb4bFZeZjmxh43pI3TC3W+AXpJ9ulFfb7BpXLXkBtVggoSw2l9aADVuN/gytObfnD5C17N3B9neOpc08D32KZRQepkXbGucWsvJJzWBvAyXF/Z4KKsUESucbWgoJSONOvMlWVZj3c8eU9CeoWrrKyoYG5ictl8zFUguU1n+kn05RveWlwm5M1HEjybzeJkMkEHMNmnwHWPPlncqGgYDPzBlNmd/Ls5jhQTBOSI5gOOQm00cJy9FdTiuNB3Q+PvF6fuXPOwhBYfOG0MZuALiT4Kg6QruQBG63C30x0grYMBOk+mKSI3oNslpnuUS5yrKYPrYjkyZcDhHGyIZOD0JeAeWry177B4oKRmkHQKB2C8R9HQViDET9Fbo1LtEtcaWKtPBbqojx25xIDv3NQW+jw8EPbrK8wI66ROGVbfPRCMpwku7POFPpgSeBZdDZYiWcqNtFYnCKdbGFAw3c4dmZ3rlNLnqHSGMIrcZKfOCWiJPhKZhrzcoW1UDPm4ZAWNcZCOBVLoy1exayJQwBOwipIZU0O63HW51JIRT7Ss6z3O904s5B+ru/HPItdn1wRarJE9Jf1/BO6f/5D756xOpB/A3LW7Uu8tzPA0Lc0cWJT5nM2EA+ajPj5hYOpqUd+RWMW4gKpnzDoL4ywk/GNuWC6ngm+qdyJPC2uYDupteWrtXdJGZzWrtSgobnDlaUjjWjxfh1HE3U20sgZGN2q7DFXvmC4tE8a26AQGS5Yxg4agzzPO5ovmlIY9HsEEevrT2aTRjyObzeRLH3pxpTSu5CRT787mU6mPiaNq5fuJpGSdh0mpOcuMKT5//Km7JVtkr2PU3ej0wOQNbzEawcNQjLqhcVnmVgrXcA9RnBTcKnodvgdj/3OJbwzSYTrUE9qmRE2HxwuaJNphdfdNjKPJ03detrpbcnEryfhuWnxIqbVUn2GTorn6oG2mfqceKTE36XOa9PmIjNrNtyxXA3TzsAEOzyaGVa8PUQ4qQadPRfp8ZfR9yBYAGSEXl9pHKNMWONeAdEIH8mAyB26wUvb/Aixa0D5Eo7w+82qKmfZkW8jtAXicrXSWG4DlauwYiSTr+ly3UKnRyfPm0BG5riDPAcGDQkL0VOTMLShPTvaIJaHqJe0o26/NXDkidFr7seQYcT2H6kkXQoGQ6+y6yOXoeY1m1KBTSEUNdCEdRGjCcwXGUmR8Do+hozPiwLSFaws75YOVhNeyc/o0Wdn/JlwmelA+ts11gZrVVoZ+mwhlXaBnokF0IIwnU3bjTEY0NEo8sFvAG/kBZB3dozfRqk1+Rasnu/Ge7Ho92R3oiX6HyKLyxcZkkDPp3gIXkLcuUGGCnAw5ubbFAilIxRXkidCMr/IwbwjLgCUSfa6mNrAjrpkD03LcLxPJqI9Js1tWwwEOAfyXVRaJ/pbubqBABifqRaqul5avh3kzuO0wW6g54KmaaIAS3XHW+5bMAwXdolrF4Gv2PQ5VCvNTs6lcAzSikzosqWJ4RtSxjjV2llJ0MLEOGuLEaDFAnNKuoK0LW+lHUQBlRaJMlYY0l1akrK9vMexl5gsHveNpcS+8Yu6asburxni4vRJJL19RjJjTKYP0PYEN2j0g3kVinLvEsgi3Q+1Zx1b5CkxnrJLgnou/Pz8nuvZrujYoue7t3Sbq3zVZoxMmcTaV8qR7RuhKniscCFceem1uqfVWUd9QW5nKvolYGTrZDGNpnbyKxtf662VW+3TKBs6yhSFcW86n75mSqjRI2xnUYU+3EuZx0QVTYdLuObx9Rg67oS/VSOdAXijmtPqhl5Upn4hkzlI54TpUz72QL6X5M63PBfPIfFrSMVF2qb29+YijW9dSEOt6BBNqNSXUPNUlVtFdwOJ+OF2lTRkzCb+/ZDAIUX06YilX1uXXQZbnF5a8on26WYUPKBOtwWRDLYhTRTBhJokEeQlDjbhfWK62erxYh6FerVXbOJe4gcRYOMJ9AH2KUPjB1QSEjPDd6zyEGUSmcYYWGH98hKGmOOMg9qTDBpLDIKb3RHxVWR9Y3rGBMJGfGn+fDcCfk8lzhkcQtI5vrJe9/DBfyuphtnyfTYnslhJPV2K/vNJSRH5yYxns9AOyEJbSUOiYIeAOhMXWk1LVhpod6h90ZHZKrFRlX640nLg7zIqyzS5DGix1C9MdYuGdM+J97YaA7pyRQ0eToHV2sDb/5fHBmuHDnTw7uP32jhBBUFCNi0sHel/iLptRj597ZLzADyjIulUDAqIC3irbGNvqFRnt1oVVCwFJCyT1dSv63UgcrzmsH5HZK9Ji6i/B0wZzXFhIS7KzSdA/560+ufYOIoBb8hzb7chBwFslIKw/kz8TDH3uWpLiDsz+WWdUyfkYSB0bdeW1S0BO02X3OJSOcao+4SDt3KGvOJjKMjiwYhfsaE8KNuqtoWOYMGG35Iuh5XsBp7v2poUfvagLQ8xFMXY4S2dhA0vSs/9CNIXrWJTb1G82KpkzZ2yQj1G6g4hH4Q43Vns5QX1FyP17mD2DXxtKEIFPmzfPz72T02dnj9+ensjj51HsR2DFRcs5Ed77fmYHuUHqAyNAwXhADS6VkzXdn3cEip00Wnzz9tXr0xOaOhcP5cTO2rblxtl9dKjIJ8Jf5WBJsxU/S/Ln8NlWGAkYGlGv9HvFLH2+C/BEkwYDMp0X71pLzbkA032tEIk8pDIdfw1LOvUmwpMMMKShI0BHPzO+wwg8uuIuYBRPxsHlXBXUofovnp+dAkpenuLXjD40G2iV27NgF034mZZMLAV+QZvqeb0CIgsNWz9tz8jFw2YaMGQZERRyvHW5MUjQ+Zt9fcY6gXtkggNGW1cRwZMlRgFRXCvXO4AR0r32+KiAPAJpFErMM/mlNeWAIBg89aYBo1+MokaGC3y9LudxPAzJJ0w9wA6dygI4ol/OnjD9+s7wrsVKUbOLB+VYCAQPUMpTeXIuvL87MDBW3GB1Q8A6nXv25G93gJOF4j0nbP+AZHBOt/K7axUg/C4Q9MT/gBP/AkPPNTnceRxRit+FcB89BBC/y2RqUUi0YOr12asT587AlIwAf4As1sfqm1oNK1qOsQSiHNQBzvodX7T4ShRrcOmHOzJIwccGnJqXJA2PQVGxUCIrjFkRI+A4XutQLvaEOo1DasodhrcJgb7uPZzRRPXA1W9HgMgea4z+LkMcmhUVMj82ScbKcNFHfHxugMEIKAyYVjqz9uFURmOWuHnIkjEenV1bJ9RSfmv3dO11jGdoXajUqSdnpy9PTvFLHUekao6OAr4u03x/RJkJVUYPKQI9o8vz/urXUChRh2SQVbM9GXJHR7KBEGeX0AM3gCEZp+pBhZdQ2VL4gzuJQouOIrDljQQrcapeyBtnCFwdlJG4108oC1ZWru8lYJ3zKl+qu0HQ9ZzhEOd6iVw+UsvkEn1HjYim3o5lAkJJFY1vIHb3mDQFyAhr2hkARsRiKaKhjjQfjlHgZX7afyxrvVufLVcvJugWmsUG2hdNhRolTt6JaqjzqlmuaMwKPAOz8ymRLJcnWaptvO45OQtAuTgAH6hlab58fn5Kz8EiNZ+/eXvy6re3t63VYsxGpaevqoC2Ps+Nz9RgGsiyc84EdhgtwCx3ZW96GXTWYzqbgQU5jwWFTqmOqiuTfqTRZz3Dw6AGrMaYrGfwS9GDtfMWRGb/6X/nwMwhtEHfO64jvUooyNNwZTLishb/w35jUwu/iGhTPpd7D0xs2gjj4unpmT13MKPrPh2pNyClB3xSkDhlmHRcHpp8fZQFISdXZzX0Y/tKoMmTmcdNDBPd47aDLjX0lTvVkN3Oc2hWZPb8lkWZLqSR0097xHCBEs/pzl2/h+q0E/tK7Kc6Qa598klcjAVbwHlhPy8lZWNDAEPjYvygW8UHdPrG+EZAg3C6WXvAK5SrZ7B9uilvMGsTHHdL6g4ihlYYOMOP+6AkxU0/KJtmI6nHEtrF0b3ZbLbob82SryeTiXf68pfnL0/1hMjPVoK88gR9V9DufjQQvxfJI6aq6e9naii9RPoG/GkDr45HqM5oYP8PCSCNaQ==", "blender_rig.py": "...", "app.py": "eNrVfdtuI8mS2Lu+Il2NsyxqiiWSurTEbvZCfZnuXkyfkbt7zjHMEWqTrCRZrWJVdV0oqQkB++IFDNg4gF8WWNjwLmDYD4sDrOEHG9i3mT+ZH7A/wRGRmXVjkaJm5qzhHoxUl6zIzMi4Z2TIMIy3QSriKf/CrsWYma9j7nphm7nCZyJJM7gZwE2QxiE9C8JUjMPwioVsEi5CFvGZF/C9KBv73oTDswD/T2OeiIAzzuBqKRL4lGUBS7MAQMD1Cz/M3KnPY2EbhrHnLaIwTtnMD8f6+lMSBvo6TPRVzAM3XOR3Ql8lYhKLNG+WZOMoDiciyZ+k81jAxIJZ/sBbiL28Y5o04wlc7e1N43DBbKbewWym3kw9FAHMVuhXM5E68sne3vlv3777wIZsZLzgC8BIbFjMeBHGsaCrD9zPXK4vU3n1nHu+vHovojAJ8eodD8IEL97wMb683Lt4/+rDq48Ie7XH4N8jWIPFOBYDZkY8CROLhZM0FsJiEw5ot1gqbtIs5m2b9bqMmsBiLQTABRiLEFYxZB+zeBwSOIMuDYB2bLHD0yOLnXThn8X63aPTtiXbvBQp933uUrtTix33+hbrHa83fM8jT7bqA7j+8YnFjtZbnQMGsM1RV3V5ut7mHb/xFtTquFvr8Kh7dgLN7vZe/avzdxffvPrgvHz7HuZmHCDpAa0evMmC24wHhy87/QOeJEAaB+KGLyJfOAB0JhJjb2/PFVPmTD1fJKbbHlCvYeyKmFZR4snhAbR3Q3vmj3FR1FPglNqT2JvZ0/FN8cQe+yJwSy3kqnhfcmjUYf5PNwOe46k3Ce1w/Kn0df54kfqlx0CIsUi8MDh07QTf1IBGnvjCEz3WSAAt6BseTzxYUn0LE3C8YBrayHhAdfj5FAlmBJ9EPJ3bn0IvMF2LTdtsGsZsyrxA4cubMt1I3HhJmpjr37RzkF8BzGkBIgFOEq6JvG/jj7VvjX2j3W6XJwb9TVEQ4ecAEGQC3U2BPd3k2kvnpmnYfjgzLN0tCIcsxsZ62ZNsseDxrYlTVmvPYbKEAWBr0+CfsiQVyImrO9m5D3xOCMmHMjX291f5Jy1iv5bFWn/eag+sO8mO+/vsh/+pWZKtuGwK9632HYtu8KVRAiiSz5nwRRqyEmBYG2xtlh7NM1xL6KwLL+SNxWh9GYimCsjSV5H6qNUBeG3su/QyEbMMpGvipCEwu5rHHUtKRDU1vg7jBR+wVWuR+am3hMXmLVyPAkzpRZsJPxGs1YPVTsPWXWVYZlPXUwRPvdLoPiq0vT0fVKfUSrxatwrDjld0G2CXjd2oxkVHZeDvgYpEPKgsAD3SiKsg5Kc//Hf20ROgEJIBo+GzzRMrcFqZDDNJBrfrU8KH+WxgLmVC2jKrvJf6zJ77NI/Gj8d+Psfmj1dma+m5Agisxb5iSRqbDVBki1a7DU1aAMMUwODyJYt8UG2EbZhTZaJrn+dTNiq8RoS5C8nu76s1ktxf7swgiT4BkQnMrTg/526bRxGgwTTOi0ZAeTAZFJFSIiGkUQXKpZJOlW780HVIAGzu5JtvX8KTmYjDtU4qYhxwf4WEvMyFStsg+XllsSXKQDmkosdL20vFIjHbTQNbcDAJnFj4nliKzYN7LxsMGLZHky1ACgZdySZoSaExmMBTn2uSNPK+jKWIQVeJxEm8wEGZA91FMDJYKUOPd0O/U+MCZdQATJcu+w2TPSzQ9mCJN8sE42iWahlpVNVdXbiMWltG0rq8Y/o1y5+21TQQvdd6rIpwQKiRaTa6bG8c/U9/+3f/+3/9ga2u7xQgpXmM74PvA7W49EkbNNH7t6/JrDPOsxRtJoPjb7T9sgXwCj6Z0xWwBZmTsccJ1fBiknE3ziLh0gcfYKQ5WvB1AKZuBiDulMJbeuJaxM48XfimO/wt0K3F5sKbzdPhCZpU+F49vvZiMfyaAwdaDMy+iKvni2RGV2ryYLe/kv4BO3zJTDSwBRgPwLsBmvreNOYLAcwfA1OXqAUoNAKo5EuA6QLKAg0idAIU9aAmd8GwoIt7zApDeSg2zgushGJZFNphRZ663hLE1a0vhi0548FK/r6Lbp64XgJy6XYw9cXNEw7cGHSIdwYTgW7Rk3UKM9Aw8Ka3HWVr6pYIoeMC7iYpiIXBJPSzRfBkxqNBrwsdjfnkahaHIK4Gj3r93mHvtAn2mAyqDrojWTLoHcGHACiMB4/O+JnL+ZMpdNqZgofh3w6SW7BQFp3Maz1rAFWeN32VeF/E4Ogkumk9+z//+b/9+6cH0OAZtnqGsseE5UWsGx+yMfpqnHQ2u2VR5iecNXUw1uDVEMWpOBOi9ey1CETMYyQLdAffngP4V5o4nx6MnxmoHgw1APqpuQ6drCz2fW8MCx0nuaf1OQO3k5p8BoaZGukQGDw10Ymz8YfZZvsoMsAaygkJSbogiM9od06NP1sOV/iiaIbkXmtm/BnxQC9vQ2yw1kgyByKPhmfSvZxIEk9onAfSs3TAKzpAT2O4qpNwq0zCoLz+fPVZja1EwpqZ4smwtYKfdy2N+mvPTedAYd3fPFknbklNg26N0pqobJ08WyBq/fAaiCfz/QSEvgiesJu4k8DwPe53wLGfXIGUadWAy8+Kj2CF5fBrRFrlTFQiHeK/QYzDfwLWOTjWnTSMBkdIsk85AwkzzecPLjRI5WELDBceXLVqBGpUKfN0fDqeTp8UbNA7RJjn49iLUV5FPCD/FkMa4COCePrpr//m6QEvqHNv7xHr/IJ/bKaYAq1zIpQEIL7zkkUorUbgNeAQlJCgS1g49SaAZNR/uSsL5mUWSJMDdCGOM4GxT7grAu4SOFMqys6z3EyEy1wttG32AhoSb7tiCQ+XQFJZIYWB2eFrFdB5xPyQLcCxgHVGpQsijsIaAI/LCdgMtQ68DpKFhz4oYnLpLUMWeNAcGZe8XYBv7/3Ft89J1Wl95FxzLzWT1GIBkAa4LnE4g4ZJWblEArsmtICygI7G/FPIfF+QHYBjpXGwv0QQf8kWIY4VLU1ozZc8mIhcr1zPgfmoK+03JunIwHuwlnIHEp+5gF/jsuB1PTATZm927bMzi5pNweqzUKtNhmCewZPWAg0K9KjKQqmDjVtpt3U5sLtTNPwLR5baJb4QEcA9zq0nhC7iuDyGmHsgCmex/SqOwxgEwtfIY+ChIHBoCz0TiUrPNgpNJOMCk8k8vHb0PJwQXW5Y28TD+B5HG4gWD0hNxv5w4bm7ADVYqGUphb0A1mSSykjJFQZc6qCVeUfNbDDYArRXhAlDf56laRjYE9+bXLVRuAPugaaSsqhbrcMbsBFOBlCLQYYr6RgUVISrJY0aNd1H7Ds1P7TWSAcRSRK5LMWXARLHRPiZz9HCAPsE6ctF6zJgC7EAmuXlIZXtJjSAjFdEcoobXGCCjyFSXJL++PdI/z/8wyESgbaMVPjoh38aAFYTNgaViuy7wW5VsutcRmaAHS9C/8c/zkKM3ClWDHJ7uFhxZRxRmEViIYK1WQ/BlOcVNQRtojZxgn4K4hVlJT5+xo7BHCfcowWoOgbrI3JoLCYo7gB4GbWIxXwxhWsS48AscyH0L4f0DrwCPC+FM55ZBKOIYNbRMuVgnFNAE1oKAbNABnfk5SzzXGRy6kpbs3ADrmAq0KIFX3INIj0FJwCgJhQPRPCAkJSDA6xDpwhSuj9Agvk1WMD8SigfdQ2wSxFSMfwYZyVxNgTKv9ASpF0SbkRFPaAKHnzhZfF2W8jm3PMBugJzPwAVQSslyuYyYZ55CS0MrR+hWj/ZKESknQeCXQaGfWXvgRSYxmjQMjMkTTEPP5H7RxGdpG0Xbl6+FEUfeAekJ6P0Nv5CKw0Du/v7hz2Qhj35tQhm0KwIn5vtxukUgOXjIX5og8HupQ7N0lynrrZ01ugDNdZPHg5KbRHYaXglAmcubswTZaul5INNQdZ07S7GVtGBupAuYSD9K1g/eCj9IJLO+Q3piwGT/lKJKJRyAfl1ia280idpF65LWuKOPkMNOYKhXuJY09wJDX0XZaoP6DexSXs06PS7JeWAD+0ojExoaVEXIBgkRU6JDoE5gXRLvpHSX7gZwG9MdQcCwg95Co1LsVZ8t5AtC5hh4AApzIQpNbdbg4xTxS/cylOlarWjjLelYV6H8ZVZgpPGt4MKgyGImHS1pIE4C362zNkme1VnZZmkhRD5G+KmrSVRLn7uhYduPlElySn40d4krUri6f5RbhFf936sRNpQrrm6o6hcIfikvEeWuBccRSqHiCHCXZs9HYI3VhKDeHE/GClah+MwBGVbyFpAl6a5ob64H5gWyARN3VS3ESrkmlOYepQ3FDcTEaXsFf1CQ4mD2V2lTr2lCB6RQHJc64TsOXIIV+ltBJ5i23Yc5AHHuQMzTtwZa58QMPooB2tLRwH0/cSsTiSKJe7zz4rXUzBTfL+BnaShC/BRY0lWzDdJ7Y90ZSonC9kTGJ0DCwWk4No2LEKcqlEUtrwhLZOSOS9ZXOG3QKz2bUHeWdXYlApFGROf3xoqGCV1qnK3leo8IDvrnbSEqtQAOi6LXGArMwHHZ5IKd6gGVmuIMeV1cCRr+f4+M7eb8uBPXQDayeMowqA//dV/re3C0dKAyGi3y3aTai4xsJu50B8UtgFYDGonF0MsClhuF5BKI8WAYge6KHkX91sGH5UpAuYx+Je0me+GT5jqmWtXVpsCpeXXoeDdCaBx7QGMUVvvvlygjzrWfP+Kp81NizWvgnzAqtdCDRdh4AlyO/O49C0r7Q8gRVSWnljyIQt/OChgW2XQ2FE8mYODk/w/X3sH5+U4a2sfl4WrpAaLoupoKCqSsPAKn6kWS7LYAlgZ09y2/az2jqEJrh6AMt2aiNdmpRRYcnPYkFrWkO6OMfPi8PCka+Q7yvS44lW1tWGF4L6gd+ViBNP+4kUS6ZRFAI/N0ZfossG1+hKpPa3RJcY+86yDzbwAXTZRtYoW1qm67Ahb1bu9moa1ylNTqFAosBfREUCWs6mzzU//8d+w/f13UuggupFC7ol27O/TtgfOuLLhXoUNWsVLQzXxXEqOeTqZm3Io0sNTNtJWnlEbCPTZPb7PkoOvn5Dbo8MyG/wSWGYYgx/iSzTnR5e5ee5ZkhJFkC2AMWCtqOt2QwDJYwdgpgaqQR48+jqkzX7AUe/uYFU0uCuFipCU0HygnbzEC4CYwfI0p+g/x4qypjYaFJtN6LhkOiM8QuqwyTYlPA/pZ5WZEA/aggeGReq/rLYAHJX2woBi8nj3mBLCgEzwrn3Hfvrr/9DwMh61AGrrsjL7HSywar//6d9t7FdaWxW+k6tb2aALZ5oKF7cO6dnELITya+4LIB30iqUOTsAZ4LGL+4fIxRhVi2KxhCZhzatuF6477jIRPeW05N6fDiOT0exvv/uIKU8qNwadG4ziSUOpRHqKGTQEL5ESsoo6FPJekBWU4y1mOwlflKMwVUdO1Y6CmUECBZ+Rb0aP2o2G+k4ZQxWpq/exF7Oa7Y141GtvwmuLrS29C05z7/jSBpfeB/PYNFBLGSxPK1KEQKBGg5PupVr8EIDK5TfFMh2gAPlAkvglT7lCI6CUltH9NZdwbwOu9AqShuLBrXk/EvP1utdh0v92XFeVWYWKGnEwAgzZXuCKm8u9Btp78AauzuScZ6nnF6EEurUnYXRbhaVvYCi05A6pWMdpr/WBEQs5QmmDrGT4pSIv1StMhbPBQwYLAyhhffDVnDlNTXVxVVIFPEn+/7Rv/rQmym5mSQXaQw0tq26BbLA65mjYQCNFiV+Qtx/Kz4S2Zp7WKgfEzHEuZsA6CLN4InIdU6VFyQhRUmEEHXIIwfzPHy4XMFrZ0F56cZpx36E9jdtSvACFO31mTzKX2/DeQYMHsWJSBKj00kscvsQk5bGP9h0hFsO53ZJybjKYqtvXK4UiNHhibyxMYUftO9S378/fgUJeLuwsES6YRxgm7g7sHtiOBw2bJAAK2sossLwxGpqvn2N61MKORIzZGHe/oTSz1xffsSzhLh80wzJno94lGK2zUReIpdb7il7We6nkxBWg8i3UhGL1fDIXMC+05YRNd7k902jL3KzlrWB6nbhBonTevP3wkQLUqQ4mz6JMXy5jvtDXpctJ0QI3Yuk63zOTgUNNa+vkhQSyRCJBORBdw4+u3V2nSxSQRZ49mZVGAHzr8U6y8FinAw5rfNuBwQ4RtPeF43xtuLckTdKi62taVGv7vhhOBW3sLBYEJgpBFtluzK+hNxkbG06SpRWEc8Ex6BuEGfB4YmwNFiZz4fsqxDThEUJ3wiyNslQ9xECCvgQ/B94NDzEA5sJVDhiTh0cymnojBfgN5UdDu9iLzLbctTDByjTaQG761jLaRaQRXRzoI7Ik1jm0sygPbh+uiRgxXs9HfXwM4oWPDi/x51EBAunBSW9SGWtUX/W6/SMi64MVfpw/kTRt3KO0SiAN3CQFrpKf3CttltxPyDqYpCaSAc4PgA2XuI0Hv3J+BcTDawUKLh31HFUDLvowjYqsvCvHkrhF8DrRsRgu8cvoysm3Gm4KkYQ5Z/n70aBz1q0EoxCglc8Xrha4EtoRqKiLwheI+RTPvejUB/OsSzGi9+GnkDbt3JBxf4bnZBKZ9OZ7uLUO7DPnt2B8+0XUrsJezYMBdOb8u7eDb6Tz3iu5NpWMstazd3JmGPLBbDi4A2lPfpLKftnmGn8Kx4l8zlOUWQEssoEPMXHe4JMUY1O4s4Z2WODgfPFOpdR/8/ZdmUDOHisKOetLEjnrStI4O1aEcHqo9uzAAZPJ+KaBBKmkIvx6dDw+Hk+7ePkbNPZM43egZQwtK7HF2fjxeDrVLfbq1q9qrlv3pmfuybgE74Xsb6L6E13ePexuhPYR5a+Ww9j+aDo+dM/w8od/fKHZ//cWewOzoVTMMyls0cIACpjMwT2txh3Q/UyrrqPcikPWAAxbLCPjkXBU0AJGEHLqL8T4kg68zMkonI86IDGeDXFhKq0wmw2menx0euQaZHbhZ2QPQIdlBw1fVD00Gm5146/UHpXkHDMMetWvohRnbeiEaJBmvwc5iDGU0zMSZtbqDejuN/AQM3NuSFK2SbZJHb5Vl6jQzU01dAPTP+sOLmsmNi1EEV14GoX+LabwsggGlibD1grGetfCmJM/bAVg12J2XhxeAautMFVty0imhmzZoTw+aN+3j6vo7dknd62DZyU/miiiNJ4k4oFm7Yi7uH8zOIxu2FmeAZhn+vWrmX4b06dXLbXWh4et6nBaj/q87x6etO5UYirMUOcZ9qDTJPTBt9kMeLIJXOvZCmnjjj0dP1sRHZKCWmV3mDH69ABnqbGAUgTdMxAzcrdfS5bcMoVOqNEz1i3IqkKIZJb6/I4MttDX6et/UuSuI5YGuQGxuulaQ5Vne9eU9gt9qBB+QpjMcdRSslgeGUKcojFLYLEd4WId06CsHKJ1tCee0tVNb9jqtthNH6j190Ddt3C/Qjbs2j287VduNSOouRwf60cdlydzHsf8dtg6YkdI4tQjmm4YSG9IL5bYePbT3/4d+0B58G5YPzdBS6yyiynPGFAnxeX3hceCWKyurOpBinro4W/+INO65JkHjZK9zYnl5fTZfrff74/XmaK6oHJU9aWvp8g2kJimwlMATA8wq5USWlmeEP6klpG9LfO2kgFfz22HiU9EZyzSa0z/pRT365hHA/xBue2UTr7CRbtb66QBw9rmQJ0Mjp+ybihvUj6pu4IgAg7qgDc7gZh+GV3L29+rZSsbMeXBLWe02/g8vAF6Zl0GxMxWb4Bk8USliJfinLIY36PbUoj1DUnXjzHFv96HTGFGFLEuIEnz0t2q1ZLUSpoFmREGs/saKbTXVgM60GBJhrXvavn1e3svPmAKLlqb49C9tZgtk9NpsTkMLGarShr4495pz2X/QnqIHNzqvYYvwCXpKHwcdwELpfZPWIUM7/Yezd2YzXtsVSRi95GCmcJUVzWJ2Kp66kE36CMqGTZLPFfUhjvuuf2jJ2yN7xR36Tc5MyEOmeYmPKOBYDG35E8BF4drj/0Qc5VUL/K22pmUHRUcKugKIbLPyqI8muE+XAkKEhmPO7RW6EWddV0xs5RhbD16PDlyp9N200JNp9Py4z1tLOVp849rC7wun2qvbzog5d3weoAs1j+C92ocx8cw8tibJeAS2UjAbKUI+47p5z4fw89VtZN+lcju9qZhmCIlaiZBVq00QIpXe4qZ57tO5mkHboNTc6G2yiKVFSMWFOWKbbpM7Oc8EWYUexhOdOaZGBpe4HozzBQMRJbG4AjT08QHq3J79AFxOxwVoF+H4cwXX8NTUxZmQJ/DyAU7arqEB0kHjXvMQbcT0O15B8jXTkEJDpmkhuJk+LbpvePy+KrUqACG5NkATdIoQmtqoMHpVnVwtJYO0Ru2InIuYJXe5oBUk+okMSiTQ5HyRc+veKVB6Pe1sRB5VSFJeZOPJ/VSXzT3VdoriLJ0G9IbG2zGOuXJO4kAGesifTUswKQ/PaRB1ttuH+l9oHOM7wB/I2onczG5Ar5X2G0YvV7y7S03Ln/ts2ZC2NioNm7J72NBSTOlKhdooV2MKGTgzMYG+cSHp8p0lAUz5LkSL52jYHiO5JKYRC9D4/yCUZ4Enj7A7X0YELH3kH7C6JJkCMq4UTB8SoaG2R4+W7nhBPzSILWRqO2Jz5PkGy8B/8V1zRbOAk+itzHe44pF6TitHtL78NoUvlg4njs0QKsatX1faPIOoIBsRu/6EasP+vtgLW5/UdmWr3T0GaN/6kAp5ULXenv46Sk8lAWj8b58zjwRu2EFnO77BZ2yNJMJmNPDQwtDAY50pQ/7XYvl80f9W0dAGc5HPq5EETe0MQ11ttHY0Lacqw6fvMVaIybR4FAmeuSZAEgTtxEQS+T5hrXz5iyj8iW0KT003r9+fm7k53j7h932Rjh6EueTCchYLwxM492Pf5+otH5mCrCVfZdjDYlY5dxg5n8bwOPeZ+OabuoDCeKepsR1wPANaHolR1LFz5Zp9/pbpp3nh4hp05Jc0GSRxuxftz9KRN/cISi5X7FDeeBjvbM3laMbtf4eBNzRdXGgi5dxGJHQuBcJI3kKw6LcfErTJ7SgDSNfyHR9eoNtdmED9SV+U8DJO8BHdFOHb1zukH3OfTTaNo1ZIfVbwESg6hIghxgP4rn8GDQz33uznLtqeTsbKEri/z16YCadB8HkLdw9lwM/l8UD6AidHOzaoZR7/+UCU1rgxnbaUFn1clwvlMo15VZZzs16wm6uXB80olJGxhAUNG6xYt4VVsnCw+keSqrwE93AQ/fHP44zn28/YLeVvsyTE1VPh/Z0xBfsrn/Yftg6n/uUUxwL7j9gjSVPSnR+ABdTxGbX7lvs2GI9+7E8LDfs2t1ejt6FSOMwedjY3uHhY5yVrgHyABFfOh2yZdFV7RCaPmXDbagVck8vNUTArPs2oKJrn+aoKDCBzlIAap67D8PGCxgifvQAJFCAKF3jRlWnLWdItCrz4V0ALqii1IOpX1O+4p6BLN1Gaf/ydOBxrw/E3zsGD7vblfVhmBmDzQ7WI+YkrmWs78oK1BEdmF0AyJgquzHzpNTPk9Ka4ol8l6vjhD+vQ5oTfS+r3NhMFoEbsGNVyM7So/HBhgrvISMYWE1ljQzc+UYtgZXm8DeWksP8c7lkmHahlnGEy3c5OrxsP3TB1IrrcwVmdNO+Z6APNJzyo2zbePBfZh4sCMYX3PvwpHCFmQ/bIOoJyZoeD+Oy38nKUWYY4cEF7oOljtYnreKDbMzSsbDaWFXFGjVYXYNENX13cfSz1E7BgL/FDfsJWO9YvXFQOn/jYZRGbq6wW6p3KVgaukijPGGValN79x5+q9ErFZXMFXxRWFJNUlWp+vF/BJRZQBDu4wl9WHDbStNqMcy+Y5g4uBniOFV6QJ7uz92j5tIvRskhm4kAbRXvC8zLn23pgnh4ewZeuQok5d/J/NT2aHB6uREuVnW52U5reF5AFo1MTDyBToGcZDgiC/GyMHE+CWizUQtX3Md3XiK97DDZ5kLOnDXMnk8wwcb7Qusv8ZYstuBtxn0J4DWaaPGtdgo+hhMsAYIJKVhdAyug+HSSo6j3+hBOkWWOkmE/9yiO0fGmGjAyg1hcK8beCHSO+VZysF+D60nnH5xJmAUpGDhYTRBWoCD6f/32opRvvwvSz5d4yB7F4GDvVxDBdMS1YpscUvKBxaqqA7O3yEzp/UwNckGlW82f/u1/6XXxnCbVCLyHv+Wh5croephNc3K0Nr6eHt9hv/Bu6PMtfdBh6Cp4XZNV/q720dd9YOXWh6BBjecFWTOgK9AM2MXO3WX9ZJECaPzbbDHGGfQPj3IMfBALT59hFROqazO8x0XOCyFs1UjnvuAplhfZxrTqTHkVwVj4FpwArE2gzN7jHOrX0IQMWFnhBE+bbIEvT5tXoHd6pwCXfmj4mHWl1aiH0jzx2MJDaZHGqLXNH/6x/essRX7cvW6nHeF4HqMk6XVpfEdH3dxKw8Cs/nAXx34Lfb0HR8/PpArdRX2qQ/cVDB6pOscwLPXa0FRfkJVyM5isJPow4+mbEMtxlI+rPchkGjv6dOI9ApYeY4wIdJxB8Sij0HJfl47Jbe5p5yhFPTxwL+RfZB7dC31N3V5QWjPQvg/Iv/dzeTSwHFG/74uHarxtTL0++t+JmGUJRZsxreKAyRzJHcyGuBjZhrn8jEi+SCYCVsbbIYb/uBy0x23yJqWNtbCzpDbKDYUL9/eVTbq/bzPAC7AQ7tyKmA7t7+inyvIBsCCZmJUP++P5a1+WWCh5BLpm+K52FG5aKOPpcYOqqe5R4IYPGPDJDnsVvaJCAhivRRGEzeJi4WDZBYnYNx/ffWOulb86/5z9+EfGwXgUEwHaoFSOagdLrF8+wy/HlEeEto0q9+R3HFRjdYSdRthUvUuNND/XtHmkOLINA23oHDhtTTQrln8JLMPjGUZygPvlLDAh7av8vD9LQHBKVbATlTWKmM3E1uC766px5Oty0G/sh3+outk//NNOimlHA2FJxfclbmgM5rpcLza/TrcYaUsH3NgGQK+1h7sdTFMMVeXMY9G8i+9AwMqk8fsCvQsdNyXyqB4pKAm+RbhJY8Gbba6hPqXQLOs/hWNluKAVaBqGqsCkqzDxKPJvHRndlInataoxMEGwEiyGCkLb+diu6mXrI0y6NQbV0lJXsgN7MufBTJjlTi2mf4+aazABJ9dL26C+ovx4ZDaLRKOl5JglJYelKMmSdGAVfGcpTUKHAYtJmIBhWajQLErcNXLZ6E9U965wtX7V4ndVm/vXroBX3hJqtspHQIHF4sil0iuA07rE4wvBJIvBSp7cFnGivcbAQ9GQjtSgW4CFikRM9vf+PlXClOBrwtdOsgkeWjPLlXgsRqMbKZopj+4h42IP6JkIWPVbEPMv6azoTUkKRcdlUWOpOnTwrHQqtNTY54sxHpo0S8UPrOJkbFsDGM1w9BS9KXElPLTlYWCzOD+vPylmWXwwLnVNRT8A8lgzKLkUlrb/L+kVfG5Jo3vDwpQOoZZg58ds9WDI1i2fDluE8rT3z0cAwQCXJhMmSFQOeta5j1AX/MYhYa134pX8RFB7v7zuMBWYjPecD6/e/+7V+2oN3inIHgfTJE38URxqe+GJGCsiyZLEVCV2InyXEzA0i1IqPTTOEoolgqUt4hSLwozjtZqxSTi5EkU5wytZOcILMEkVN2CuxbhsUWEMRY+VqhpeNRSBcLGOEZ5mr9Yyaix7QkCBAsJE1ByZzUce187rp/VRGWkWBAL3dovB4dHp5oGltqoXm+oxICoiREVMShAxZTHC11fsqNuuZXNJHNryl3QBkh2mndhjD4tiGF2b/sOYVoMZqivCNiHn2w9UMWcddqVqiIYA41ek5fMsmMwducw0veHj0xM9MZrocI3+cg6Cd0WmbsFXEqhJyfkxlc8blqamHlNfEp+yd3V0Fw+biE0ZERioFkHqyAJ4Dibz5Z+F145AHKgHOm+m1xTPpLC3cB1K7B+O6qUByhsVSmBoqhpplqDSmcj666KAeHaS/+U0N89ivhGVP35VamJsqcUBn5VLd1aPcoOImglK1el8pp/f4u7NzZZsEmOeplEyODiYAclmY3sSLkojKQ/qAMwJwRORHGB+dJIeoP+OgvdgsztulL7vAKKymw5fuCdHKEEp2bNWOAXmOpmD7MZZWqwbPj4+rkhXeFwlVZBD+OfUEpDMoEVgshkQ1NDgeGhT02+vWmRKN68W+MAna+Vms9hP+FSYZ+1/TgYo0zzjWTofmjgrKx9nw942NnMWQAZYbbMpk3V3XvlF7CEVI3mt0pXpHy/GRp1pUHtUOAYQDbdUpHpNLko+qXJQuUzXpFrR4IKKvoyIgAp5b3Q6QdjBP9QiC5/IR9DtLn74lJgEeKTXf0zr1huscKnudgljy4oDw/II3168QhsVnIu4/PzDx5eA1FLlglLReSzsUJRsK/QMFcivFnNL2VN20q0V2pKHAbEnVQLBRjLwq7U/yADF+n4CTEGMWJhxLh5GvPOl2znrXH71vQ3LU6zG9ygyMOqJXeBfIjHW6iAt1hWRXPCFjanjkdngvYNNwouKrJtKnGqTb0RltFE9O6ieS/O83FIBtUqUilYuFZ52OacvC7gaxZ/FrJ3GR+GbWxk45bVCjgrCd4n+CwgiwFpXTP5hzrBU3p/+bGcsE35D27aNmglct5WaBdKaUCJZIwVBs6jZLDkaSf/hurQgBxoYjQglb8MSrckNpRagtZT8xcD3/i+9a7Kq", "estudio.html": "eNq1O11v20iS7/4VjDITkglFfVmOLZnKJo4nycLZZOPMzgKGkTTJlsSYZHNJSpasCNhfcC8H3N3bYR7ugHs44IB72Ld72PyT/QX3E66qu0k2JdmTmdkbICOyu7q6ur6r2D6+5zMvXyZUm+ZRONo7xh8tJPHEadCsMTqeUuKPjiOaE82bkjSjudOY5ePmYQOA+XBMIuo05gG9TliaNzSPxTmNAew68POp49N54NEmf7GCOMgDEjYzj4TU6SCOPMhDOjrN8pkfMK33/LglRvaOs3yJv4OUsXzVbLqTwf1Ot9PrHA6bzYTENIR3t+N39+E9DGI6uN8lXb93AK/5Ih/cp4f0iFJ4i2bwdkSOfELgjXje4H7f7bvjNryxK8AyPvIP3PXew5XLFs0suAniycBlqU/TJoyskSeWy/zlKiLpJIgH7eGUBpNpPui0298OXeJdTVI2i/3BnKQGUmoOPRayVL4DNeZwDGwZdPaThZYts5xGzVlgNUmShLQpBqxzOmFU+/6VlZE4a2Y0DcZDNqfpOGTXg2ng+zRe793PWbJKWAZ8ZPGAuBkLZzkdwigQkyyGIR3n4imVFMKjH2RJSJaDcUgXwwlJBocwSMJgEjcD2DkbeCAwmg5vmkHs08WgP0TI5nUKoPi/9Z6d0cmqhmbr1Fwk5lDwbdDBg7Iw8DUxiwIqJpsp8YNZJmhLiO8jv3uJIK2bLMR2mjvLcxZbtpvHK2W3PAX2JCQFkmtcBimXu7c3dnqsbATPWgd2ERIJ4ikwGjDN0gxQJSzgnLieAmOasI1HBzGreCCJslmNpPtdrzvukS2h19YMgH3EDam/YoA2yJcDu9cv9vXpmMzCHBckK+TvoAPPvl+JOqUhyYM55cOjTaZ8vQh20Oj7mh3ReFYKOGYxHW4rmdCotlC2zi7N/5U6cADPURALZzHoHOFcRBZNaW0H7fm0sggyyxkn3mYJjTdO4IbMu4JZHNSIJX6FHOowQ7kXGnJOF3mTm8UAzWjbhjmATz2WEs4ZwSdJ/CHqFVJcPyCeSeETX1Jq6R0qWBI/mOKRa0cQQzs0sFiURSQMV1vWAXwj+YCLETwJXSS36BG7KsWk4sBhpLh5XcijDWhcBiRFO3ySmBhwW+NuqY88RusdszQSdgw6Tf9oNGHGVLxPzdFwV+QHKfU4eiBoFsW7vFfh2FBjhFSPDubXQCGJgyhbbTlBlEzdzw0/zbI8GC+bMoZJzAWKQoF+td0prqpQk+6GHfyki9ogyt5FF4S6nYLk42Lg/ng8BlRBPGY7RCiiCQQtbvMHSOO2UqFGQNCkQtKFFPeHeO7CdDt2H3YJ6YTG/uo2z8LXS615vGH7+/1N21etSiYCnveLXU9pvRvHqbnE8mzlWTQ/mO/WrB0KKrMHQKu11wWCYFV4IFhUphVbbqSn0BLEnLXSx90Ho97F0yCGZA18TI26HVTt1vltOReH7633PBLPSbbpbH8TUT8gmlHZ38E+HMRc7Wk8ayk4g6p0WPiEwzJTOcTAr21FftUmjury6fEVQncRqdAKoT+7N1BdvDyngq9T4dPctBYNcVw6OrlBry2gc8bCbFX3OaWiNhcyTGl1D1Icqr9D6+SpspxMqEv8Cd06W6ff3uBaYVk1gM2TrfeOWzKnPs68NEhyDTN/pxFEmLlHJGmMVvIlawxWjXyaUtoYNKZ5nmSDVsvzY/tT5tMwmKd2TPNWnEQtDvSbtt05aNvtljsLQl+M2RHzZyGFFQ1LoGrBqVmctX4eSrogEWTJWetTFrUa6zUcghMPp2iJ6gRTc3gDU9QC32kAu6Cw0DQ+4IUky5wGaFWDT2KdkvFpABDS0HySk+YcFtJFo1jA4sboPSjLLCXHLQE32oSHJBQYNXob0BuSlVA7UYuIicA0Y9mtGGHzZWP0NPWCMCQqxuMWHGbrUHDOnRO+Lw4b4qaNchc5C1bVGP3tX/6snc1utL/90/9U1CgoMI3YxaYQ60JeqzWKom33aRBwAQYWsBQg5ZNmgHMy72AVrIIfcHwehWVPi8dbt/BTEkEu5pHG6Dk8fvkRn2+FBnOOAfIt/mhGCpVpRlItF2KuyBI8vZOzHomi3Yz933/9h3/TTr78CM6e/Gzuek6jbbUbo++wuqG7zwEwnUMOdQr1SeiTW8GOONQZ8ZkW3PzJvoPtANxUoCHu2Ldi7fWtDlT/vdb+rSBt67CNapwG7u3UjaG2aoxOiGpgP814SFp38/10gX4LxHknzwscQgC1/Yqfwo1gWOX0TUgMPAFvRkP2tz//+yZc5agbGnevQOzddRT6834Vl2ppU7+W1sg2RZWqbaeIO9LIejAZ8qbGFOR6PWhrba2DqY7EfHhYepDq3DyUbI9jWNweFRGn8rc4xkPdLa6pdMQ8bm7YAI6nLK854acQ+JowSLadADfEHPzECVhLqgBsIb2G8gHsnoVf/nPC4h0OmAvyioZoVH+a0ZDmbDfMFMgDG//Hv2gnJKlFh8pLFz+1ICuiIRxYBFjtoUYy7f3Ld6en2jhlkabzqKcPi/nVm9QN8hMQZQqMWqtARRz15GSrBgrxVMHy4uz9d2egyVAw7EIR8qmsVYHVlz9/9/TkzVesV+DqCN4xFp3G8yBlMdhcvhMJreaz1sYCgW0Pjprl2jdO5ox85s34DMgpXZ6DrLycpUZmDiXUd6/OTs+dFbj1wYUuzPbDNXXtSejqVjGAOgomUR8UkSC4KSYuLR7iAU3CY7wclMEchzGay1GM3TBERPAWg+uCpt+9Ojl1VpvUDPQXZ880iNI0ZRoMagYEMY9pj7Tfvj19Ye6mViySI5oByyztBfNZbmnved71KVNWgouxx+4CVn337I/Ym9VooeCa8X0c5EtL+z5OKQkt7XUAqRaDxXuaXG67IfgCWPwMfynEbzFi3s4zQV4WKBtVwJA6YLRmNnM/AeSbZ78FqJxHbaZs+wG0J6UZOM2eb2d5CKDn7880Ofrlv2Ot9xzqjEghI2R+50O/LUk4e/O8YGu//W0Nqvuh01egotmygOz0v+U0KLIe6CK500CZtekMpA24KqEP9NckIRAuNT4Gc6r0gWjw2xo2OwhMzQOfsg+QOAQxSe0o2QeAP+CYkKWH5yrBJqD/PUh/VTB4/+t/cQjg4ocE0xdiJ/EEAF5FEINiJIRPamJSX5eWk3Lx0dSJ6bXwOvYP1H1x9k6OGysSY2c+INkgT2fUQk7TdE5BIa8h3jybjcdQVuPUGgytQAfVWv42WNDwHTbEjNckn9pwPkM0/asZq2uaQ3XROUQnI4hjmv4gPwzA40tuVyp6NsuTGXg2CH/n2It1BOnn7148qwYrxOCGKUgkAYIl5NOT0/PvgjAKvPfVnEIJD4swbNOY92YdPOFwr/QxWFrYsAoWnEyhtDHKpT64qZAiUOl4MigpqcLhc3w3zCEft6u4roDwUxjthfiwUWJKopRGCtjb1+9OX78ALClBX1cQgU0cBAcHCrWKw1fZ6GDFzrh+w50apgU11b5pC7OlxYaglup24NIT7LfN6QmJYFMDkr5KWi1FWoiuY3Xb7ZJ2Lw85qlpQMmADayfvAFwy/znUeSg5LgIcxpr3HUT+nO4aO08oyKtjd4uNJ2mgsvYFvL6kYQLK3Wlb+5DdLrq01xEP3e5+1yskA2HIwMXlEaB0Z6qpvKbZ1FDYA/UDfUFZRPN0aXTbcHzTUsQulQpLHhIaart/bfImLEttzGgwK7QXTpObzdtXrW4xByUPsF7gEUevCOUgJaVTGgUKoS/hNUumwOQzFA4o1pj/B0fuwX/7+5Z9oJ4al1fqO4sVVM+LjisJN3B1AcMstj2S5TUSYUzYkx2RBA0cDR3Ys39o4f/UfQG23DYNoru3PTwkBLbtqAhgESDYg+iieTMQ1SwMrQjcjXwkHEXmrNYQlpl3pVocvoMZQFasjD5jix4MvgR9OrQwERSaI7Bh5ouoyJwEQrfP4WDlAUQOxMerNMoobNPHiM4nlRwJZvk4cug59SA4pW9BCQz9l/ZFwsDNWhxlaxLm45ZuDgVZfAdlYw6DrBvPYs4kjTcJjCts02mC5DNnJWv8wUUldeuia+1bvUurkghok9Wx2xbK/9Iqyn25qOv7Vs+GZX0LFACXuWP/EJbZfcs+wqViGSbrZc0v1pL2ARgVQFw092Fjvvig/biNewJGMCT+T6wvWwAKsT3Y46Jn9a1mB9eOx/3+48cWytYGojt9a0zCDNbyhoCyzt63LtqwrC9WCdOBdQd8X7FofXF1OQSiuQ2g+0YWv6QL4+yifSlMAxv0MVR9S+fsonPJh4oqkNuEbdtnF91L045mYR4k4fLcIyFJjZcQJINoE2vvUgyrWPc5CThaQ9zsPnxp4b9m7yEgQ/OuLetfShtSUm4Yfnz5RISQAer7NsgrBcUB3zkYG/dwnbnasccj+wC7jaWGuWBrhrkCQ7VnCZT/EIfzNFj8wFIIqChE7MwvwGtc0dMoyZel7bhbFop7Ix4o9+YQpKjBnNEKiGF2kKGXNouX8ysMUr4YAzu74ploLAkARAzYHEFyQZ9hSMYUB4kcujCcLAGFWw2bmAGES0H1PsxFFf3mmhebFDQDtpnIoPATuEuwr94E+TOLgZmGa67XmEtB9AbOsoXC5zFYAjXIjQO2QUOn0+U2LbgvOeo5iGlC8xPe/VaC2h94GQUsNq2sAOJZ2i6Q4UtH5HpkYWT20oIcQIhGuDx1cmEhQGbfmA879lGLz+QkxoTAHrP5wyL4QXZrQgQskRCH3JST2PCiDg3VAQ5KNgzAsxePOAwUIAYxBbzHMoOaD33Ls5fVrBy5eVTCbMKLI0HSAd4JuCGkh1/OYNeYktTxW5D48Ncxvj3sixehY29T9kmEMiFTQ+Q6pQKqFoJlUQYmgvEsd9rW3GkP71LzBw/uMXsGCfpz8Jw2djfAvEQS5FTqNZw/ciY2yWF3F5Qxq1jlgd7lwxynedvpifwVE63e4I5lrR4qYCmnr1AWgP3G0Pl3Fd20RfL4/vWZ8/GbFec3T4qNHJJSdsbwZtA5bB1PDHOtwcOXH+PJLGSZ9te/aN+s5juA5l/+I4UAQLNjNx19swKdA6DvIBvwjS5Mf/lnDQeXuwZvaoPRx0dozQZSavNyL3vyUeyrDK1FJZh9HOg6mD/JlrGnlaKkC6jAMmNsrkACUiipQ65JAHklzb2pMbZWIJ4p8wf6y9Onz/XKnKHguVp7BIFAoHKQR531em9zo2zKrg38nKFEbo/EfubwBsgFTl0OUaPGQUh5JjOEWtSQKa7GxgLcBL3imY09JUi3ueLw46GbUnK1ls4ex0xBUSFPTDBAnrznacuWp6PjZ0a9Uo+JPLrMRvDnKZ7DQIRg01ytKUghJ6GpYMX65ETeJPu43YTVaqrTabcfUo6b+q0C2frbj0JPYQOwJXMlwhqUSGxO+cgaZqpcbwNAmRhuJIRrEYTAePiK4d0BiW1lymxXhs+gcJtB1hWdwA5Q2HCxD3lwAXQoTMdxdGwy6SYPDry62ChSzsGz+pBKVcWHaB1D7nXket6BBfyaTGOaZQO737fw/l4o3tp9wazabrLRpeMhvnpHYAMIjxe2spegbPp4DeFL7FQl8yiL4WakimLuWKIgtpecw6UXWjadKJalUjUIHhNruPoIIK2qgmFVFQxR03ibuuaQdF3qC7g/7MlwcDuk8SQHQUpM5fGfFiCvcaI8RaH5vC/vXOjvKJCEfaMT0frBJ5ZCGQwP5ySc+UQ+5fzhGVgif3hNYt5Segm1MR94yzL64b1+KTapkZixNDcMYrmmMzL4xsKfvwHTtvGCpvno6Mj89uiouTHr1mYl/ZWX8NBLqDuZK8nEC4+vvHQ4V2wvDJKnfAYDJMdS8MF1ym6KBx4lp7L0N3TRONcxMVLNXWAGGwGH4lFD/wBn1zgUi2EfqOYMOCZ6G0OAmjVxqt0aV6ZnHLigvJDI5ZPiaaAeEbJ4iVUcTQZs7HNwbOhRwVOeg0+QRnqb/as5aLFCdtCQYPQqQG9xLdC5J2HUSf4h5AwiSuGWdM4wNB+a/wBx30BgTABqwPhlgARxJsDLCCw+7Wz57E1z//y5GOHtb/2Jzm946AMdP2jpW9gqA9rE9EQ/dstP3iMMzyfEJxp3SpqjzWIRSm2cec5AJ+FYWkRv4CSxpYVEw3tsBAd9BlZg6wMuzk3ycJPiIzzfRX8kYrho5X7+DIH6ka6JNwu2JUpDdyCsXiTPUDbzchhEeiaakTI1G94e8Uqm7P4y8TQMYSm/d6AVGg+Zb3pKIMy7zshV5JazySQUQrZcG7+UgpjtOZyVB3qeNMqeB3of4daqNJKreVykgpBBS42PL4cYw0kRwEVERAwPHsiHe45DTPkM2axP38ygTu2ZQ2JjJzg3TD76KsZBm29kDgsyyE+dXr388rWnV12C48Rb/sA01QRaWgOLeQV4D52xPKxgxnXmXFzeYqn1RPoaEt5ZNjWYCXYKLwWx187o2hb3jgtjvDZvLUjvFQnA58+bG1RZFI81W9Wq8TRNyRJG+a9RhV7zSfU8uKieLyuORs4o4rtwdXaAHTK3XJfblfpxvdnhrCoHqz7zjGSBt51THLqH7nhslduJaK/czxYDSv/T8mmST39Ig5wORF5rltHius4mkRVd26Jp/IYH0y5kSZgwXGMNvC7bf9xSHV32rHSlfb+tisXl6U0tLEILZqQULJwlUL8lZMJjQtXMc1ypAjKIyXHm+Lu9L4Qi4a7vJkmhZuGMFrvcPsdkcjtmprobMkROY2JVbgTjp3N4QCD8cmDo/IS6hdHz70SOeSevL8S9m8vbeI2ErEr5Vf4u3HTC66/Zx/uJfeofFETkRqeesvzOQBtgllSQ5mHYwZsrkBP/7MaKBM2d7UnPXliiB7aA/PblQ7v9GHsTRUK52ebIATy3l/Dv5hEA9/uit6D2KfKNdkPlBgQVF8SC7E05mQ3hLICszAJuRCQxfjeLXCx+ZFfJoj8lhjtjnCoLXrkq4Q3VqJLEXVK7V3/fKT81itTBzbXYx8vTjX2K4L+nZFN1OoSZx869r8i3hjthVLrQLyshS9Ilc747Nt7KCndsvAWzufFeGQfUurb6fCVbpfLbGS9rKsAy6Njclb8HpyttaRcM3lMWfwbWVb/+KKW10O87C+hfGQ9Xka2EJBFWIlsGJYd/lOPRRDZxv7YJMPz/JrqgsDOsH0DGTaS50JvptuGU31rFg8GPZYEjKbwQubUsIzpmfVMI6s6OL7agThijv393ZugBXjto4RUEXOGz6xgzZEf3xNUofjkBJjhZBmr5dkzCCx03VASlFfo5wr88O7u/Nt/dXv3aCwabNxjKQv6MscRQzM6HYhQrH/TrzykU6IYsCh88qNWG/razvY37uPeewVt5fCNQO+wT8k+MolNmqL1CPQ0mH3gp8ynjpRz/BV6WXcLKo0OpVH6cfOMic+w5CaHGEbUQSb1pMGdQDa1k1rXRDfyVl6Us9eKMekumfivm0gQ6xZfsXURIwq/oMjPw0pS5AS6YA8KPggy8CcSdC9vG30setGSPVFijgC0bsqbocgpckgW/NJxVRbtotirR7NLOIIk2gIZaV1VpmUcOWq28hap/HRe2mrQ/w4rHqnHiS1WwI/KL8eWjj9ox/5ux0TercZEI2HDoBAzCxCv3fPIjuCK1vUK429ygnZOJ9N6TyH8drWoZON65v8D+e6774JbOKWr6WwIhPTPAeonMmnCU8xqrhrnzezRqQ5/r5ufPeo49ayRZts5vk+HcUSXDgUExgtg3rpyRWHt121rYhndvZfyV+4s0AbPKjm7uzBtEfi+TUXnfJy/IpzlJkHvYRM8LRrtcv6p7yVsdNprDwEYXQ7R41qWJic8L2HEwMTEq/+riuMX/3OK4Jf5g/P8A0jGTig=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()